# 🛒 Brazilian E-Commerce — Executive Sales Dashboard

An interactive, multi-view **Plotly Dash** dashboard built on the Olist master dataset
(`data/processed/brazilian_ecommerce_data.csv`).

**What you get**

| Section | Content |
|---|---|
| **KPI strip** | Sales, Orders, Average Order Value, Avg Review Score, Avg Delivery Time, On-Time Delivery % |
| **Key insights** | Auto-generated business takeaways that update with every filter |
| **Sales trend** | Monthly sales + order volume + 3-month moving average |
| **Category analysis** | Top-10 categories by sales and a price-vs-satisfaction bubble matrix |
| **Behaviour** | Weekday × hour order heatmap |
| **Customer experience** | Review score distribution, delivery time vs review score, on-time delivery trend |

**Filters:** year (multi-select), product category (multi-select) and a one-click reset.

> Run the notebook top-to-bottom, then open the link printed by the last cell (`http://127.0.0.1:8070`).

In [ ]:
# If a package is missing, uncomment and run once:
# %pip install -q dash plotly pandas numpy

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import dash
from dash import dcc, html
from dash.dependencies import Input, Output

warnings.filterwarnings("ignore")
pd.options.display.float_format = "{:,.2f}".format

## 1 · Configuration
The loader tries several locations, so the notebook works both on your machine and from the repo folders.

In [ ]:
# ── Data location ───────────────────────────────────────────────────────────
DATA_CANDIDATES = [
    Path(r"E:\Courses\depi\depi-mini-and-final-projects\mini-projects\data\processed\brazilian_ecommerce_data.csv"),
    Path("../data/processed/brazilian_ecommerce_data.csv"),     # run from /notebooks or /Dashboard
    Path("data/processed/brazilian_ecommerce_data.csv"),        # run from /mini-projects
    Path("brazilian_ecommerce_data.csv"),                       # file next to the notebook
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "brazilian_ecommerce_data.csv not found. Edit DATA_CANDIDATES in this cell with the correct path.")
print("Using data file:", DATA_PATH)

# ── Design system ───────────────────────────────────────────────────────────
C = dict(
    navy="#0F172A", slate="#475569", muted="#94A3B8", line="#E2E8F0", bg="#F1F5F9", card="#FFFFFF",
    blue="#2563EB", blue_light="#93C5FD", teal="#0D9488", amber="#F59E0B", rose="#E11D48", green="#16A34A",
)
REVIEW_COLORS = {1: "#DC2626", 2: "#F97316", 3: "#FACC15", 4: "#84CC16", 5: "#16A34A"}
FONT = "Inter, -apple-system, Segoe UI, Roboto, Helvetica, Arial, sans-serif"
CURRENCY = "R$"
MIN_MONTHLY_ROWS = 500          # months with fewer order-lines are treated as incomplete and excluded
EXCLUDED_STATUSES = ["canceled", "unavailable"]
WEEKDAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

## 2 · Load & prepare the data
* Cancelled / unavailable orders are excluded (no realised revenue).
* Dates are parsed, and delivery metrics are derived (`delivery_days`, `on_time`).
* Months at the edges of the dataset with very few records (e.g. Sep-2016, Dec-2016, Sep-2018) are dropped so they do not distort trends.
* Rows are **order items**, so order-level metrics (reviews, delivery) are always computed on de-duplicated `order_id`s.

In [ ]:
USE_COLS = ["order_id", "order_status", "product_category", "order_purchase_timestamp",
            "order_delivered_customer_date", "order_estimated_delivery_date",
            "price", "freight_value", "review_score"]

raw = pd.read_csv(DATA_PATH, usecols=USE_COLS, encoding="utf-8-sig")
for col in ["order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date"]:
    raw[col] = pd.to_datetime(raw[col], errors="coerce")

df = raw[~raw["order_status"].isin(EXCLUDED_STATUSES)].dropna(subset=["price", "order_purchase_timestamp"]).copy()

# time features
df["month_start"] = df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()
df["year"]        = df["order_purchase_timestamp"].dt.year
df["weekday"]     = df["order_purchase_timestamp"].dt.day_name()
df["hour"]        = df["order_purchase_timestamp"].dt.hour

# drop sparse edge months
rows_per_month = df.groupby("month_start").size()
valid_months   = rows_per_month[rows_per_month >= MIN_MONTHLY_ROWS].index
dropped_months = sorted(set(rows_per_month.index) - set(valid_months))
df = df[df["month_start"].isin(valid_months)].copy()

# category label
df["category"] = (df["product_category"].fillna("unknown").str.replace("_", " ").str.title())

# delivery features
df["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.total_seconds() / 86400
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan
delivered_mask = df["order_delivered_customer_date"].notna()
df["on_time"] = np.where(delivered_mask,
                         (df["order_delivered_customer_date"] <= df["order_estimated_delivery_date"]).astype(float),
                         np.nan)

YEARS = sorted(df["year"].unique())
CATEGORIES = sorted(df.loc[df["category"] != "Unknown", "category"].unique())

print(f"Rows: {len(df):,} | Orders: {df['order_id'].nunique():,} | Categories: {len(CATEGORIES)}")
print(f"Period: {df['month_start'].min():%b %Y} → {df['month_start'].max():%b %Y} | Years: {YEARS}")
print("Dropped sparse months:", [f"{m:%b %Y}" for m in dropped_months])

## 3 · Metric helpers
Everything the dashboard shows is computed by small, testable functions.

In [ ]:
def apply_filters(years, cats):
    # Empty selection = no filter (i.e. 'All').
    d = df
    if years:
        d = d[d["year"].isin(years)]
    if cats:
        d = d[d["category"].isin(cats)]
    return d

def orders_of(d):
    # One row per order (reviews and delivery are order-level facts).
    return d.drop_duplicates("order_id")

def money(x, short=True):
    if x is None or pd.isna(x):
        return "–"
    if short:
        for div, suf in [(1e9, "B"), (1e6, "M"), (1e3, "K")]:
            if abs(x) >= div:
                return f"{CURRENCY} {x / div:,.2f}{suf}"
    return f"{CURRENCY} {x:,.2f}"

def num(x, nd=0, suffix=""):
    return "–" if x is None or pd.isna(x) else f"{x:,.{nd}f}{suffix}"

def compute_kpis(d):
    o = orders_of(d)
    sales = d["price"].sum()
    return dict(
        sales=sales,
        orders=o["order_id"].size,
        aov=d.groupby("order_id")["price"].sum().mean() if len(d) else np.nan,
        review=o["review_score"].mean(),
        delivery=o["delivery_days"].mean(),
        on_time=o["on_time"].mean() * 100 if o["on_time"].notna().any() else np.nan,
        freight_pct=(d["freight_value"].sum() / sales * 100) if sales else np.nan,
    )

def build_insights(d):
    # Plain-language takeaways that react to the current filters.
    if d.empty:
        return ["No data for the current selection."]
    out, sales = [], d["price"].sum()

    cat = d[d["category"] != "Unknown"].groupby("category")["price"].sum().sort_values(ascending=False)
    if len(cat):
        out.append(f"**{cat.index[0]}** is the #1 category with {money(cat.iloc[0])} "
                   f"({cat.iloc[0] / sales:.1%} of sales).")
        if len(cat) >= 5:
            out.append(f"The top 5 categories generate **{cat.head(5).sum() / sales:.0%}** of sales.")

    m = d.groupby("month_start")["price"].sum()
    if len(m):
        out.append(f"Peak month: **{m.idxmax():%B %Y}** with {money(m.max())} in sales.")

    o = orders_of(d)
    dv = o[o["on_time"].notna() & o["review_score"].notna()]
    if len(dv) and (dv["on_time"] == 0).any() and (dv["on_time"] == 1).any():
        late, ok = dv[dv["on_time"] == 0]["review_score"].mean(), dv[dv["on_time"] == 1]["review_score"].mean()
        out.append(f"**{(dv['on_time'] == 0).mean():.1%}** of delivered orders arrive late — and they score "
                   f"**{late:.2f}★** vs **{ok:.2f}★** for on-time orders.")

    k = compute_kpis(d)
    out.append(f"Shipping costs equal **{k['freight_pct']:.1f}%** of product sales.")
    return out

## 4 · Figure builders
One function per chart. All share the same typography, colours and hover style.

In [ ]:
def style(fig, title, subtitle=None, height=380, legend=False):
    t = f"<b>{title}</b>" + (f"<br><span style='font-size:12px;color:{C['muted']}'>{subtitle}</span>" if subtitle else "")
    fig.update_layout(
        title=dict(text=t, x=0.0, xanchor="left", font=dict(size=16, color=C["navy"])),
        template="plotly_white", height=height, font=dict(family=FONT, size=12, color=C["slate"]),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=10, r=10, t=70, b=10), showlegend=legend,
        hoverlabel=dict(bgcolor=C["navy"], font=dict(color="white", family=FONT, size=12), bordercolor=C["navy"]),
    )
    fig.update_xaxes(showgrid=False, linecolor=C["line"], zeroline=False)
    fig.update_yaxes(gridcolor=C["line"], zeroline=False)
    return fig

def empty_fig(title, height=380):
    fig = go.Figure()
    fig.add_annotation(text="No data for the current selection", showarrow=False,
                       font=dict(size=14, color=C["muted"]), xref="paper", yref="paper", x=0.5, y=0.5)
    fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
    return style(fig, title, height=height)

def fig_trend(d):
    title = "Sales Trend"
    if d.empty:
        return empty_fig(title, 420)
    m = d.groupby("month_start").agg(sales=("price", "sum"), orders=("order_id", "nunique")).reset_index()
    m["ma3"] = m["sales"].rolling(3, min_periods=1).mean()
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(x=m["month_start"], y=m["orders"], name="Orders", marker_color=C["blue_light"], opacity=0.45,
                hovertemplate="%{x|%b %Y}<br>Orders: %{y:,}<extra></extra>", secondary_y=True)
    fig.add_scatter(x=m["month_start"], y=m["sales"], name="Sales", mode="lines+markers",
                    line=dict(color=C["blue"], width=3), marker=dict(size=7),
                    hovertemplate=f"%{{x|%b %Y}}<br>Sales: {CURRENCY} %{{y:,.0f}}<extra></extra>", secondary_y=False)
    fig.add_scatter(x=m["month_start"], y=m["ma3"], name="3-month avg", mode="lines",
                    line=dict(color=C["amber"], width=2, dash="dash"),
                    hovertemplate=f"%{{x|%b %Y}}<br>3M avg: {CURRENCY} %{{y:,.0f}}<extra></extra>", secondary_y=False)
    style(fig, title, "Monthly sales (line) vs. number of orders (bars)", height=420, legend=True)
    fig.update_layout(legend=dict(orientation="h", y=1.12, x=1, xanchor="right"), hovermode="x unified")
    fig.update_yaxes(title_text=f"Sales ({CURRENCY})", tickprefix="", secondary_y=False)
    fig.update_yaxes(title_text="Orders", showgrid=False, secondary_y=True)
    return fig

def fig_category(d):
    title = "Top 10 Categories by Sales"
    t = d[d["category"] != "Unknown"].groupby("category")["price"].sum().nlargest(10).sort_values()
    if t.empty:
        return empty_fig(title)
    total = d["price"].sum()
    fig = go.Figure(go.Bar(
        x=t.values, y=t.index, orientation="h",
        marker=dict(color=t.values, colorscale=[[0, C["blue_light"]], [1, C["blue"]]]),
        text=[f"{money(v)}  ·  {v / total:.1%}" for v in t.values], textposition="outside", cliponaxis=False,
        hovertemplate=f"%{{y}}<br>Sales: {CURRENCY} %{{x:,.0f}}<extra></extra>"))
    style(fig, title, "Share shown as % of total sales in the current selection")
    fig.update_xaxes(visible=False, range=[0, t.max() * 1.35])
    fig.update_yaxes(showgrid=False)
    return fig

def fig_matrix(d):
    title = "Category Performance Matrix"
    o = d[d["category"] != "Unknown"]
    g = o.groupby("category").agg(sales=("price", "sum"), avg_price=("price", "mean"),
                                  review=("review_score", "mean"), orders=("order_id", "nunique")).reset_index()
    g = g.nlargest(15, "sales").dropna(subset=["review"])
    if g.empty:
        return empty_fig(title)
    fig = go.Figure(go.Scatter(
        x=g["avg_price"], y=g["review"], mode="markers+text", text=g["category"], textposition="top center",
        textfont=dict(size=10, color=C["slate"]),
        marker=dict(size=g["sales"], sizemode="area", sizeref=2.0 * g["sales"].max() / (46 ** 2), sizemin=8,
                    color=g["review"], colorscale=[[0, C["rose"]], [0.5, C["amber"]], [1, C["green"]]],
                    line=dict(color="white", width=1.5), opacity=0.85),
        customdata=np.stack([g["sales"], g["orders"]], axis=-1),
        hovertemplate=("<b>%{text}</b><br>Avg price: " + CURRENCY + " %{x:,.2f}<br>Avg review: %{y:.2f}★"
                       "<br>Sales: " + CURRENCY + " %{customdata[0]:,.0f}<br>Orders: %{customdata[1]:,}<extra></extra>")))
    style(fig, title, "Top 15 categories · bubble size = sales · colour = avg review")
    fig.update_xaxes(title_text=f"Average item price ({CURRENCY})", showgrid=True, gridcolor=C["line"])
    fig.update_yaxes(title_text="Average review score")
    return fig

def fig_heatmap(d):
    title = "When Do Customers Buy?"
    o = orders_of(d)
    if o.empty:
        return empty_fig(title)
    p = (o.groupby(["weekday", "hour"]).size().unstack(fill_value=0)
          .reindex(index=WEEKDAYS, columns=range(24), fill_value=0))
    fig = go.Figure(go.Heatmap(
        z=p.values, x=[f"{h:02d}:00" for h in p.columns], y=[w[:3] for w in p.index],
        colorscale=[[0, "#EFF6FF"], [0.5, C["blue_light"]], [1, C["navy"]]], xgap=2, ygap=2,
        colorbar=dict(title="Orders", thickness=10, len=0.8),
        hovertemplate="%{y} %{x}<br>Orders: %{z:,}<extra></extra>"))
    style(fig, title, "Orders by weekday and hour of day")
    fig.update_yaxes(autorange="reversed", showgrid=False)
    fig.update_xaxes(tickangle=-45, dtick=2)
    return fig

def fig_reviews(d):
    title = "Customer Satisfaction"
    r = orders_of(d)["review_score"].dropna().astype(int).value_counts().reindex(range(1, 6), fill_value=0)
    if r.sum() == 0:
        return empty_fig(title)
    fig = go.Figure(go.Bar(
        x=[f"{i}★" for i in r.index], y=r.values, marker_color=[REVIEW_COLORS[i] for i in r.index],
        text=[f"{v / r.sum():.1%}" for v in r.values], textposition="outside", cliponaxis=False,
        hovertemplate="%{x}<br>Orders: %{y:,}<extra></extra>"))
    style(fig, title, f"Review score distribution · {r.sum():,} reviewed orders")
    fig.update_yaxes(visible=False, range=[0, r.max() * 1.18])
    return fig

def fig_delivery(d):
    title = "Delivery Time vs. Review Score"
    o = orders_of(d).dropna(subset=["review_score", "delivery_days"])
    if o.empty:
        return empty_fig(title)
    g = o.groupby(o["review_score"].astype(int))["delivery_days"].mean().reindex(range(1, 6))
    fig = go.Figure(go.Bar(
        x=[f"{i}★" for i in g.index], y=g.values, marker_color=[REVIEW_COLORS[i] for i in g.index],
        text=[num(v, 1, "d") for v in g.values], textposition="outside", cliponaxis=False,
        hovertemplate="%{x}<br>Avg delivery: %{y:.1f} days<extra></extra>"))
    style(fig, title, "Slower deliveries are strongly linked to lower ratings")
    fig.update_yaxes(title_text="Avg delivery days", range=[0, np.nanmax(g.values) * 1.2])
    return fig

def fig_ontime(d):
    title = "On-Time Delivery Rate"
    o = orders_of(d).dropna(subset=["on_time"])
    if o.empty:
        return empty_fig(title)
    m = o.groupby("month_start").agg(rate=("on_time", "mean"), n=("order_id", "size")).reset_index()
    m["rate"] *= 100
    fig = go.Figure(go.Scatter(
        x=m["month_start"], y=m["rate"], mode="lines+markers", fill="tozeroy",
        line=dict(color=C["teal"], width=3), marker=dict(size=7), fillcolor="rgba(13,148,136,0.12)",
        customdata=m["n"], hovertemplate="%{x|%b %Y}<br>On-time: %{y:.1f}%<br>Orders: %{customdata:,}<extra></extra>"))
    style(fig, title, "Share of delivered orders arriving on or before the estimated date")
    fig.update_yaxes(range=[max(0, m["rate"].min() - 10), 100], ticksuffix="%")
    return fig

## 5 · Layout & styling
A clean card-based layout: dark header, sticky filter bar, KPI strip, insights panel and a responsive chart grid.

In [ ]:
CSS = '''
* { box-sizing: border-box; }
body { margin: 0; background: #F1F5F9; font-family: Inter, -apple-system, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; color: #0F172A; }
.header { background: linear-gradient(120deg, #0F172A 0%, #1E3A8A 100%); color: white; padding: 26px 40px; }
.header h1 { margin: 0; font-size: 28px; font-weight: 800; letter-spacing: -0.5px; }
.header p  { margin: 6px 0 0; color: #BFDBFE; font-size: 14px; }
.container { max-width: 1480px; margin: 0 auto; padding: 22px 28px 40px; }
.filters { position: sticky; top: 0; z-index: 50; background: #FFFFFF; border: 1px solid #E2E8F0; border-radius: 14px;
           padding: 14px 18px; display: flex; gap: 18px; align-items: flex-end; flex-wrap: wrap;
           box-shadow: 0 4px 16px rgba(15,23,42,.06); margin-bottom: 20px; }
.filter-box { display: flex; flex-direction: column; gap: 6px; }
.filter-label { font-size: 11px; font-weight: 700; letter-spacing: .08em; text-transform: uppercase; color: #64748B; }
.btn-reset { background: #0F172A; color: #fff; border: 0; border-radius: 8px; padding: 10px 18px; font-weight: 600; cursor: pointer; font-family: inherit; }
.btn-reset:hover { background: #2563EB; }
.kpi-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(190px, 1fr)); gap: 16px; margin-bottom: 20px; }
.kpi { background: #fff; border: 1px solid #E2E8F0; border-radius: 14px; padding: 18px 20px; border-top: 4px solid var(--accent, #2563EB);
       box-shadow: 0 1px 3px rgba(15,23,42,.05); }
.kpi-label { font-size: 11px; font-weight: 700; letter-spacing: .08em; text-transform: uppercase; color: #64748B; }
.kpi-value { font-size: 28px; font-weight: 800; margin: 8px 0 2px; letter-spacing: -0.5px; }
.kpi-sub   { font-size: 12px; color: #94A3B8; }
.card { background: #fff; border: 1px solid #E2E8F0; border-radius: 14px; padding: 8px 14px 4px; box-shadow: 0 1px 3px rgba(15,23,42,.05); }
.insights { background: #fff; border: 1px solid #E2E8F0; border-left: 5px solid #F59E0B; border-radius: 14px; padding: 16px 22px; margin-bottom: 20px; }
.insights h3 { margin: 0 0 8px; font-size: 14px; letter-spacing: .06em; text-transform: uppercase; color: #64748B; }
.insights ul { margin: 0; padding-left: 18px; line-height: 1.8; font-size: 14px; color: #334155; }
.grid-2 { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 16px; margin-bottom: 16px; }
.grid-1 { display: grid; grid-template-columns: 1fr; gap: 16px; margin-bottom: 16px; }
.section-title { font-size: 12px; font-weight: 700; letter-spacing: .1em; text-transform: uppercase; color: #64748B; margin: 26px 0 10px; }
.footer { text-align: center; color: #94A3B8; font-size: 12px; margin-top: 30px; }
@media (max-width: 1000px) { .grid-2 { grid-template-columns: 1fr; } .header { padding: 20px; } }
'''

app = dash.Dash(__name__, title="Brazilian E-Commerce Dashboard", suppress_callback_exceptions=True)
app.index_string = (
    "<!DOCTYPE html><html><head>{%metas%}<title>{%title%}</title>{%favicon%}{%css%}"
    "<link rel='preconnect' href='https://fonts.googleapis.com'>"
    "<link href='https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700;800&display=swap' rel='stylesheet'>"
    "<style>" + CSS + "</style></head><body>{%app_entry%}<footer>{%config%}{%scripts%}{%renderer%}</footer></body></html>"
)

GRAPH_CFG = {"displayModeBar": False}

def graph(gid):
    return html.Div(dcc.Loading(dcc.Graph(id=gid, config=GRAPH_CFG), type="dot", color=C["blue"]), className="card")

app.layout = html.Div([
    html.Div([
        html.H1("🛒 Brazilian E-Commerce · Executive Sales Dashboard"),
        html.P(f"Olist marketplace · {df['month_start'].min():%b %Y} – {df['month_start'].max():%b %Y} · "
               f"{df['order_id'].nunique():,} orders · all values in Brazilian Real ({CURRENCY})"),
    ], className="header"),

    html.Div([
        html.Div([
            html.Div([
                html.Div("Year", className="filter-label"),
                dcc.Dropdown(id="f-year", options=[{"label": str(y), "value": int(y)} for y in YEARS],
                             value=[], multi=True, placeholder="All years", style={"minWidth": "220px"}),
            ], className="filter-box"),
            html.Div([
                html.Div("Product category", className="filter-label"),
                dcc.Dropdown(id="f-cat", options=[{"label": c, "value": c} for c in CATEGORIES],
                             value=[], multi=True, placeholder="All categories", style={"minWidth": "380px"}),
            ], className="filter-box"),
            html.Button("↺ Reset filters", id="btn-reset", n_clicks=0, className="btn-reset"),
        ], className="filters"),

        html.Div(id="kpis", className="kpi-grid"),
        html.Div(id="insights", className="insights"),

        html.Div("Sales performance", className="section-title"),
        html.Div([graph("g-trend")], className="grid-1"),
        html.Div([graph("g-category"), graph("g-matrix")], className="grid-2"),

        html.Div("Customer behaviour & experience", className="section-title"),
        html.Div([graph("g-heatmap"), graph("g-reviews")], className="grid-2"),
        html.Div([graph("g-delivery"), graph("g-ontime")], className="grid-2"),

        html.Div("Built with Python · Pandas · Plotly · Dash  |  DEPI Data Science Mini-Project", className="footer"),
    ], className="container"),
])

## 6 · Callbacks
A single callback refreshes every KPI, insight and chart from the same filtered slice, keeping all views consistent.

In [ ]:
def kpi_card(label, value, sub, accent):
    return html.Div([
        html.Div(label, className="kpi-label"),
        html.Div(value, className="kpi-value"),
        html.Div(sub, className="kpi-sub"),
    ], className="kpi", style={"--accent": accent})

def render_bold(text):
    # Turn **bold** markers into <strong> elements.
    parts = text.split("**")
    return [html.Strong(p) if i % 2 else p for i, p in enumerate(parts)]

@app.callback(
    Output("f-year", "value"), Output("f-cat", "value"),
    Input("btn-reset", "n_clicks"), prevent_initial_call=True)
def reset_filters(_):
    return [], []

@app.callback(
    Output("kpis", "children"), Output("insights", "children"),
    Output("g-trend", "figure"), Output("g-category", "figure"), Output("g-matrix", "figure"),
    Output("g-heatmap", "figure"), Output("g-reviews", "figure"),
    Output("g-delivery", "figure"), Output("g-ontime", "figure"),
    Input("f-year", "value"), Input("f-cat", "value"))
def update_dashboard(years, cats):
    d = apply_filters(years, cats)
    k = compute_kpis(d)

    cards = [
        kpi_card("Total sales", money(k["sales"]), "Sum of item prices", C["blue"]),
        kpi_card("Orders", num(k["orders"]), "Unique orders", C["teal"]),
        kpi_card("Avg order value", money(k["aov"], short=False), "Sales per order", C["amber"]),
        kpi_card("Avg review", num(k["review"], 2, " ★"), "Out of 5.0", C["green"]),
        kpi_card("Avg delivery time", num(k["delivery"], 1, " days"), "Purchase → delivery", C["rose"]),
        kpi_card("On-time delivery", num(k["on_time"], 1, "%"), "Delivered by estimated date", C["navy"]),
    ]
    insights = [html.H3("💡 Key insights")] + [html.Ul([html.Li(render_bold(t)) for t in build_insights(d)])]

    return (cards, insights, fig_trend(d), fig_category(d), fig_matrix(d),
            fig_heatmap(d), fig_reviews(d), fig_delivery(d), fig_ontime(d))

## 7 · Run the dashboard

In [ ]:
if __name__ == "__main__":
    try:
        app.run(port=8070, host="127.0.0.1", debug=False, jupyter_mode="external")   # Dash >= 2.11
    except TypeError:
        app.run_server(port=8070, host="127.0.0.1", debug=False)                       # older Dash

## 8 · Metric definitions

| Metric | Definition |
|---|---|
| **Sales** | Sum of `price` for every order item (freight excluded) |
| **Orders** | Count of unique `order_id` |
| **Average order value** | Mean of per-order total `price` |
| **Avg review** | Mean `review_score` per unique order |
| **Delivery time** | `order_delivered_customer_date − order_purchase_timestamp` in days (delivered orders only) |
| **On-time %** | Delivered orders with delivery date ≤ `order_estimated_delivery_date` |
| **Freight share** | Total `freight_value` ÷ total `price` |

**Data rules:** cancelled/unavailable orders are excluded; sparse edge months (< 500 order lines) are removed; items without a category are labelled *Unknown* and hidden from category charts.